# Transportation Network Optimization

Objective:Determine the lowest-cost allocation of customer demand across available plants while respecting plant capacity constraints.
This baseline model focuses on facility allocation decisions and establishes the foundation for future scenario analysis and network resilience studies.

In [1]:
# Import optimization and analysis libraries

import pandas as pd
import numpy as np

from pulp import (
    LpProblem,
    LpMinimize,
    LpVariable,
    lpSum,
    LpStatus,
    value
)

In [2]:

file_path = "../data/raw/Supply chain logistics problem.xlsx"

order_list = pd.read_excel(file_path, sheet_name="OrderList")
freight_rates = pd.read_excel(file_path, sheet_name="FreightRates")
wh_costs = pd.read_excel(file_path, sheet_name="WhCosts")
wh_capacities = pd.read_excel(file_path, sheet_name="WhCapacities")
products_per_plant = pd.read_excel(file_path, sheet_name="ProductsPerPlant")
vmi_customers = pd.read_excel(file_path, sheet_name="VmiCustomers")
plant_ports = pd.read_excel(file_path, sheet_name="PlantPorts")

In [3]:
# Create plant capacity table from source data

plant_capacity = wh_capacities.copy()

# Create customer demand table

customer_demand = (
    order_list
    .groupby('Customer')['Unit quantity']
    .sum()
    .reset_index()
)

customer_demand.columns = [
    'Customer',
    'Demand'
]

# Create plant summary table

warehouse_cost = wh_costs.copy()

plant_summary = (
    plant_capacity
    .merge(
        warehouse_cost,
        left_on='Plant ID',
        right_on='WH',
        how='left'
    )
)

In [4]:
# Create scaled demand so the model remains feasible

total_capacity = plant_capacity['Daily Capacity '].sum()

customer_demand_scaled = customer_demand.copy()

customer_demand_scaled['Scaled_Demand'] = (
    customer_demand_scaled['Demand']
    / customer_demand_scaled['Demand'].sum()
    * total_capacity * 0.95
)

customer_demand_scaled.head()

,Customer,Demand,Scaled_Demand
0,V555555555555555555_17,266457,49.669102
1,V555555555555555555_42,470632,87.728485
2,V555555555555555555_45,116136,21.648412
3,V555555555555555555_46,12080,2.251781
4,V555555555555555_23,375,0.069902


In [5]:
print("Total Capacity:", total_capacity)

print(
    "Total Scaled Demand:",
    round(customer_demand_scaled['Scaled_Demand'].sum(), 2)
)

Total Capacity: 5791
Total Scaled Demand: 5501.45


In [6]:
# Create plant list

plants = plant_summary['Plant ID'].tolist()

# Create customer list

customers = customer_demand_scaled['Customer'].tolist()

In [7]:
# Create capacity dictionary

capacity = dict(
    zip(
        plant_summary['Plant ID'],
        plant_summary['Daily Capacity ']
    )
)

# Create cost dictionary

warehouse_cost_dict = dict(
    zip(
        plant_summary['Plant ID'],
        plant_summary['Cost/unit']
    )
)

# Create demand dictionary

demand = dict(
    zip(
        customer_demand_scaled['Customer'],
        customer_demand_scaled['Scaled_Demand']
    )
)

In [8]:
# Create minimization model

model = LpProblem(
    "Transportation_Network_Optimization",
    LpMinimize
)

In [9]:
# Create shipment decision variables

x = LpVariable.dicts(
    "shipment",
    [(i, j) for i in plants for j in customers],
    lowBound=0
)

# Minimize total warehouse operating cost

model += lpSum(
    warehouse_cost_dict[i] * x[(i, j)]
    for i in plants
    for j in customers
)

# Demand satisfaction constraints

for j in customers:

    model += (
        lpSum(
            x[(i, j)]
            for i in plants
        )
        >= demand[j]
    )

# Capacity constraints

for i in plants:

    model += (
        lpSum(
            x[(i, j)]
            for j in customers
        )
        <= capacity[i]
    )

    # Solve optimization model

model.solve()

print("Status:", LpStatus[model.status])


Status: Optimal


In [10]:
# Extract shipment results

results = []

for i in plants:
    for j in customers:

        qty = x[(i, j)].varValue

        if qty and qty > 0:

            results.append([
                i,
                j,
                qty
            ])

results_df = pd.DataFrame(
    results,
    columns=[
        'Plant',
        'Customer',
        'Shipment'
    ]
)

results_df.head()

,Plant,Customer,Shipment
0,PLANT15,V555555555555555555_45,11.000000
1,PLANT17,V5555555555_1,1.200639
2,PLANT17,V55555555_5,6.799361
3,PLANT05,V55555555555_28,352.763190
4,PLANT05,V555555555_35,0.841063


In [11]:
# Calculate objective value

print(
    "Total Network Cost:",
    round(value(model.objective), 2)
)

print(
    "Plants Utilized:",
    results_df['Plant'].nunique()
)

Total Network Cost: 3300.75
Plants Utilized: 18


In [12]:
# Calculate plant utilization

plant_usage = (
    results_df
    .groupby('Plant')['Shipment']
    .sum()
    .reset_index()
)

plant_usage = plant_usage.merge(
    plant_capacity,
    left_on='Plant',
    right_on='Plant ID'
)

plant_usage['Utilization %'] = (
    plant_usage['Shipment']
    / plant_usage['Daily Capacity ']
    * 100
)

plant_usage.sort_values(
    by='Utilization %',
    ascending=False
)

,Plant,Shipment,Plant ID,Daily Capacity,Utilization %
9,PLANT10,118.000002,PLANT10,118,100.000002
7,PLANT08,14.000000,PLANT08,14,100.000002
8,PLANT09,11.000000,PLANT09,11,100.000001
4,PLANT05,385.000005,PLANT05,385,100.000001
0,PLANT01,1070.000004,PLANT01,1070,100.000000
1,PLANT02,138.000000,PLANT02,138,100.000000
3,PLANT04,554.000000,PLANT04,554,100.000000
5,PLANT06,49.000000,PLANT06,49,100.000000
12,PLANT13,490.000000,PLANT13,490,100.000000
13,PLANT14,549.000000,PLANT14,549,100.000000


In [13]:
plant_analysis = (
    plant_usage
    .merge(
        warehouse_cost,
        left_on='Plant',
        right_on='WH'
    )
)

plant_analysis[
    ['Plant', 'Shipment', 'Daily Capacity ',
     'Utilization %', 'Cost/unit']
].sort_values(
    by='Shipment',
    ascending=False
)

,Plant,Shipment,Daily Capacity,Utilization %,Cost/unit
0,PLANT01,1070.000004,1070,100.000000,0.566976
2,PLANT03,1012.999992,1013,99.999999,0.517502
3,PLANT04,554.000000,554,100.000000,0.428503
13,PLANT14,549.000000,549,100.000000,0.634330
12,PLANT13,490.000000,490,100.000000,0.469707
4,PLANT05,385.000005,385,100.000001,0.488144
10,PLANT11,331.999997,332,99.999999,0.555247
15,PLANT16,278.450000,457,60.929978,1.919808
6,PLANT07,264.999998,265,99.999999,0.371424
11,PLANT12,208.999997,209,99.999999,0.773132


## Baseline Optimization Results

The optimization model allocated demand according to facility operating cost while respecting plant capacity constraints.

Key findings:

- Nearly all plants were utilized at or near full capacity.
- Lower-cost facilities were prioritized by the optimizer.
- Higher-cost facilities were used only when lower-cost capacity was exhausted.
- PLANT16 retained unused capacity because its operating cost was significantly higher than most alternatives.

These results indicate that facility operating cost is a major driver of network allocation decisions when transportation costs are excluded.

In [14]:
plant_analysis[
    [
        'Plant',
        'Daily Capacity ',
        'Cost/unit',
        'Shipment',
        'Utilization %'
    ]
].sort_values(
    by='Cost/unit'
)

,Plant,Daily Capacity,Cost/unit,Shipment,Utilization %
6,PLANT07,265,0.371424,264.999998,99.999999
3,PLANT04,554,0.428503,554.000000,100.000000
16,PLANT17,8,0.428947,8.000000,100.000000
8,PLANT09,11,0.465071,11.000000,100.000001
12,PLANT13,490,0.469707,490.000000,100.000000
1,PLANT02,138,0.477504,138.000000,100.000000
4,PLANT05,385,0.488144,385.000005,100.000001
9,PLANT10,118,0.493582,118.000002,100.000002
2,PLANT03,1013,0.517502,1012.999992,99.999999
7,PLANT08,14,0.522857,14.000000,100.000002
